# Free-Form KAN Contraction Discovery — Linear Study

**Question:** Can a generic multi-layer B-spline KAN — given only the raw
normalised stiffness contrast `T` and the strain `ε` per voxel, with **no
polarization-identity scaffold** — discover that the target function
`τ(T, ε) = T:ε` is bilinear?

This notebook trains `FreeFormKANTauTheta` via **standalone pointwise
regression** (no unrolled LS loop), which is ~50× cheaper than embedded
training and isolates the function-discovery question cleanly.

| | Scaffolded `KANTauTheta` (Study 2) | Free-form `FreeFormKANTauTheta` (this) |
|---|---|---|
| Combination formula | Hard-coded `s, a, b` + polarization identity | None — raw `(T, ε)` fed to a generic KAN |
| Learnable object | One 1D function `φ` (3 ctrl pts) | Full multi-layer KAN (~2,430 params) |
| Training mode | Embedded LS-FNO, K=140 steps | Standalone regression, instant per step |
| Session limit concern | ~60 min @ 40 epochs | **~5–10 min total** — no concern |

Checkpoints are saved to Drive every 500 steps so that a second session can
resume from where the first left off (though training finishes in one session).

In [ ]:
# §1 — Colab runtime check
import sys, subprocess
import time
IN_COLAB = 'google.colab' in sys.modules
print('Running in Colab:', IN_COLAB)

if IN_COLAB:
    from google.colab import userdata
    print('Colab secrets available.')

In [ ]:
# §2 — Clone repo and install package
import os
from pathlib import Path

GITHUB_REPO = 'MiRoSi-52wab/Training'
CLONE_DIR   = '/content/ls-kan-fno'

if not Path(CLONE_DIR).exists():
    try:
        token     = userdata.get('GITHUB_TOKEN')
        clone_url = f'https://{token}@github.com/{GITHUB_REPO}.git'
        print('GITHUB_TOKEN found in Colab secrets.')
    except Exception:
        clone_url = f'https://github.com/{GITHUB_REPO}.git'
        print('Warning: GITHUB_TOKEN not set — will fail for private repos.')
    result = subprocess.run(['git', 'clone', clone_url, CLONE_DIR], capture_output=True, text=True)
    if result.returncode != 0:
        raise RuntimeError(f'Clone failed: {result.stderr}')
    print('Cloned successfully.')
else:
    subprocess.run(['git', '-C', CLONE_DIR, 'pull'], check=True)
    print('Pulled latest changes.')

subprocess.run([sys.executable, '-m', 'pip', 'install', '-e', CLONE_DIR, '-q'], check=True)
os.chdir(CLONE_DIR)
REPO_ROOT = CLONE_DIR
print(f'Package installed. CWD = {os.getcwd()}')

In [ ]:
# §3 — Mount Drive and configure paths
if IN_COLAB:
    from google.colab import drive
    drive.mount('/content/drive')

# ── Edit these if your Drive layout differs ───────────────────────────────────
DATA_PATH   = '/content/drive/MyDrive/ls_kan_fno/data/dataset_v3.h5'
RUN_DIR_FF  = '/content/drive/MyDrive/ls_kan_fno/runs/freeform_kan'
# ─────────────────────────────────────────────────────────────────────────────

Path(RUN_DIR_FF).mkdir(parents=True, exist_ok=True)
assert Path(DATA_PATH).exists(), f'Dataset not found at {DATA_PATH}'
print(f'Dataset : {DATA_PATH}')
print(f'Run dir : {RUN_DIR_FF}')

In [ ]:
# §4 — GPU check
import torch
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device : {device}')
if device.type == 'cuda':
    print(f'GPU    : {torch.cuda.get_device_name(0)}')
    print(f'VRAM   : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
else:
    print('No GPU — training will still finish in a few minutes (CPU is fine for this model).')

In [ ]:
# §5 — Build model
from utils.config_loader import load_config, compute_alpha_bounds
from models.freeform_kan import FreeFormKANTauTheta
from datasets.micromechanics import DataLoaderFactory

cfg = load_config('configs/freeform_kan_colab.yaml')
cfg['train_path'] = DATA_PATH
cfg['val_path']   = DATA_PATH
cfg['output_dir'] = RUN_DIR_FF

# Compute α₀ = (α⁻ + α⁺)/2 from material parameters (same formula LSFNO uses internally).
# For the default experiment.yaml (kappa=10, E_matrix=1.0, nu=0.3, 2D), α₀ = 10.0.
_am, _ap = compute_alpha_bounds(
    cfg['E_matrix'],
    cfg.get('nu_matrix', 0.3),
    cfg.get('nu_inclusion', cfg.get('nu_matrix', 0.3)),
    cfg['kappa'],
    dim=int(cfg.get('dim', 2)),
)
ALPHA0   = (_am + _ap) / 2.0
N_NORMAL = 2   # 2D plane strain: 2 normal + 1 shear Mandel component

torch.manual_seed(int(cfg.get('seed', 42)))

model = FreeFormKANTauTheta(
    n_comp          = 3,
    width           = cfg.get('width', [18]),
    grid_size       = int(cfg.get('grid_size', 5)),
    k               = int(cfg.get('k', 3)),
    x_range         = tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale = float(cfg.get('eps_input_scale', 100.0)),
).to(device)

print(model)
print(f'\nLayer param counts: {model.layer_param_counts()}')
print(f'Total params      : {model.n_params():,}')
print(f'\nPhysics constants  α₀={ALPHA0:.4g}  (α⁻={_am:.4g}, α⁺={_ap:.4g}), n_normal={N_NORMAL}')
print(f'eps_input_scale    = {model.eps_input_scale}  (ε features are scaled up before KAN)')

In [ ]:
# §6 — Data helpers: Voigt → Mandel conversion and target ξ computation
from models.freeform_kan import (
    compute_T_mandel, voigt_strain_to_mandel, contraction_loss
)

# Random ε sampling range (from config).
# ε★ from the dataset is CORRELATED with T (both derive from the same microstructure
# via the FFT solver).  Training on (T, ε★) pairs lets the model learn a shortcut
# that fits the correlation manifold instead of the true bilinear law T:ε.
# The bilinearity diagnostic then fails (std/mean ≈ 1, cosine ≈ 0.1).
#
# Fix: sample ε RANDOMLY and independently of T.  Since ξ = T:ε is computed in
# closed form, no FFT data is needed for ε.  The model must learn the true bilinear
# law to minimise the loss on arbitrary (T, ε) pairs.
eps_scale = float(cfg.get('eps_scale', 0.02))   # ε ~ Uniform[-eps_scale, eps_scale]
                                                 # × eps_input_scale=100 → KAN sees [-2, 2]

def prep_batch(batch):
    """
    T_M from real microstructures (realistic T distribution).
    eps_M sampled uniformly at random — independent of T.
    xi_true = T:eps_M computed in closed form (exact label, no FFT needed).
    """
    C_V = batch['C_field'].to(device).double()
    T_M = compute_T_mandel(C_V, ALPHA0, N_NORMAL)      # (B, n, n, H, W)

    B, n, _, H, W = T_M.shape
    eps_M = (torch.rand(B, n, H, W, device=device, dtype=torch.float64) * 2 - 1) * eps_scale

    xi_true = torch.einsum('bijhw,bjhw->bihw', T_M, eps_M)
    return T_M, eps_M, xi_true

# Quick shape sanity check
loaders   = DataLoaderFactory.from_config(cfg)
sample_b  = next(iter(loaders['train']))
T_, e_, xi_ = prep_batch(sample_b)
print(f'T_M    : {tuple(T_.shape)}')
print(f'eps_M  : {tuple(e_.shape)}   (random, independent of T)')
print(f'xi_true: {tuple(xi_.shape)}')
print(f'T range: [{T_.min():.3f}, {T_.max():.3f}]')
print(f'ε range: [{e_.min():.5f}, {e_.max():.5f}]   (uniform in [-{eps_scale}, {eps_scale}])')
print(f'ξ range: [{xi_.min():.5f}, {xi_.max():.5f}]')
print(f'ε×{int(model.eps_input_scale)} KAN input: [{e_.min()*model.eps_input_scale:.2f}, {e_.max()*model.eps_input_scale:.2f}]'
      f'  (x_range = {cfg.get("x_range", [-2.0, 2.0])})')

In [ ]:
# §7 — Smoke test: 2 steps, verify forward + backward, check shapes
import time

print('Running 2-step smoke test...')
model.train()
opt_smoke = torch.optim.Adam(model.parameters(), lr=1e-3)
losses = []
for _ in range(2):
    b = next(iter(loaders['train']))
    T_s, e_s, xi_s = prep_batch(b)
    t0 = time.time()
    opt_smoke.zero_grad()
    xi_pred = model(T_s, e_s)
    loss = contraction_loss(xi_pred, xi_s)
    loss.backward()
    opt_smoke.step()
    dt = time.time() - t0
    losses.append(float(loss))
    print(f'  step  loss={float(loss):.4e}  time={dt*1000:.0f}ms/step')

import math
assert all(math.isfinite(l) for l in losses), 'NaN/Inf detected in loss!'
assert xi_pred.shape == (b['C_field'].shape[0], 3, 64, 64), f'Bad output shape: {xi_pred.shape}'
print('\nSmoke test PASSED ✓')
print('Expected initial loss ≈ 1.0 (random init, no signal yet)')
print(f'Estimated full run: ~{dt * int(cfg.get("n_steps", 5000)) / 60:.0f} min on this device')

In [ ]:
# §8 — Adam training with checkpoint save / resume
#
# Loss = L_task + λ × L_sparse
#   L_task   = relative L2  ‖ξ_pred − ξ_true‖ / ‖ξ_true‖
#   L_sparse = L1 penalty on edge function magnitudes (λ = sparsity_lambda in YAML)
#            = Σ_{layers} Σ_{edges (i,j)} mean_batch |φᵢⱼ(x)|
#   Setting sparsity_lambda=0 recovers pure L2 loss.
#
# With width=[18] and 9 inputs: 36/162 L1 edges should be active (22%).
# L_sparse pushes the other 126 toward zero, making the sparse solution findable.
#
# Checkpoint is saved to Drive every save_every steps.

import json
from itertools import cycle

# Re-build model fresh (undo smoke-test parameter drift)
torch.manual_seed(int(cfg.get('seed', 42)))
model = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)

n_steps       = int(cfg.get('n_steps', 5000))
log_every     = int(cfg.get('log_every', 100))
val_every     = int(cfg.get('val_every', 500))
save_every    = int(cfg.get('save_every', 500))
lambda_sparse = float(cfg.get('sparsity_lambda', 0.0))

optimizer = torch.optim.AdamW(
    model.parameters(),
    lr=float(cfg.get('lr', 1e-3)),
    weight_decay=float(cfg.get('weight_decay', 0.0)),
)
scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=n_steps, eta_min=float(cfg.get('lr_min', 1e-5))
)

CKPT_PATH = Path(RUN_DIR_FF) / 'freeform_kan_checkpoint.pt'
HIST_PATH = Path(RUN_DIR_FF) / 'freeform_kan_history.jsonl'

print(f'Loss = L_task + {lambda_sparse:.1e} × L_sparse'
      + ('  (sparsity disabled)' if lambda_sparse == 0 else '  (L1 sparsity active)'))

# ── Resume if checkpoint exists ────────────────────────────────────────────
start_step = 0
history    = []
if CKPT_PATH.exists():
    ckpt = torch.load(CKPT_PATH, map_location=device, weights_only=False)
    model.load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    start_step = int(ckpt['step']) + 1
    history    = ckpt.get('history', [])
    past_vals  = [r['val_loss'] for r in history if 'val_loss' in r]
    best_past  = min(past_vals) if past_vals else float('nan')
    print(f'Resumed from step {start_step - 1}  (best val so far: {best_past:.4e})')
else:
    print(f'Starting fresh training for {n_steps} Adam steps.')

# ── Training loop ─────────────────────────────────────────────────────────
train_iter = cycle(loaders['train'])
hist_fh    = open(HIST_PATH, 'a')

t0 = time.time()
model.train()

for step in range(start_step, n_steps):
    T_M, eps_M, xi_true = prep_batch(next(train_iter))

    optimizer.zero_grad()
    xi_pred   = model(T_M, eps_M)
    loss_task = contraction_loss(xi_pred, xi_true)

    if lambda_sparse > 0.0:
        # Flatten inputs for sparsity evaluation (same preprocessing as forward)
        x_kan, _, _ = model._flatten_inputs(T_M, eps_M)   # (B*N_sp, 9)
        # Subsample 1024 voxels to keep overhead small (~1-2% of total step time)
        n_sub = min(1024, x_kan.shape[0])
        idx   = torch.randperm(x_kan.shape[0], device=device)[:n_sub]
        loss_sparse = model.sparsity_loss(x_kan[idx].detach())
        loss = loss_task + lambda_sparse * loss_sparse
    else:
        loss_sparse = torch.zeros(1, device=device)
        loss = loss_task

    loss.backward()
    optimizer.step()
    scheduler.step()

    row = {
        'step':       step,
        'train_loss': float(loss),
        'task_loss':  float(loss_task),
        'sparse_loss': float(loss_sparse) * lambda_sparse,
        'lr':         scheduler.get_last_lr()[0],
    }

    if step % val_every == 0:
        model.eval()
        val_l = []
        with torch.no_grad():
            for vb in loaders['val']:
                T_v, e_v, xi_v = prep_batch(vb)
                val_l.append(float(contraction_loss(model(T_v, e_v), xi_v)))
        row['val_loss'] = sum(val_l) / max(len(val_l), 1)
        model.train()

    history.append(row)
    hist_fh.write(json.dumps(row) + '\n')
    hist_fh.flush()

    if step % log_every == 0:
        elapsed = time.time() - t0
        done    = step - start_step + 1
        eta     = (n_steps - step) * (elapsed / max(done, 1))
        val_str = f"  val={row.get('val_loss', float('nan')):.3e}" if 'val_loss' in row else ''
        sp_str  = f"  sp={row['sparse_loss']:.2e}" if lambda_sparse > 0 else ''
        print(f'Step {step:5d}/{n_steps}  task={float(loss_task):.3e}{sp_str}{val_str}'
              f'  lr={row["lr"]:.1e}  ETA {eta/60:.1f} min')

    if (step + 1) % save_every == 0 or step == n_steps - 1:
        torch.save({
            'step':                 step,
            'model_state_dict':     model.state_dict(),
            'optimizer_state_dict': optimizer.state_dict(),
            'scheduler_state_dict': scheduler.state_dict(),
            'history':              history,
            'config':               cfg,
        }, CKPT_PATH)
        print(f'  ✓ Saved checkpoint at step {step} → {CKPT_PATH.name}')

hist_fh.close()
elapsed_total = time.time() - t0
steps_done    = n_steps - start_step
print(f'\nAdam phase done: {steps_done} steps in {elapsed_total:.0f}s'
      f' ({elapsed_total / max(steps_done, 1) * 1000:.1f} ms/step)')

In [ ]:
# §9 — LBFGS refinement phase
#
# Runs a single LBFGS pass on a small fixed batch after Adam plateaus.
# Sharpens the B-spline coefficients.  Typically < 30 s.
#
# Skip this cell if Adam loss is already below 1e-3.

lbfgs_steps = int(cfg.get('lbfgs_steps', 200))
lbfgs_bsize = int(cfg.get('lbfgs_batch_size', 4))

lbfgs_loader = torch.utils.data.DataLoader(
    loaders['train'].dataset, batch_size=lbfgs_bsize, shuffle=True
)
lb_batch = next(iter(lbfgs_loader))
T_lb, eps_lb, xi_lb = prep_batch(lb_batch)
print(f'LBFGS on {lbfgs_bsize} samples × 64×64 = {lbfgs_bsize*64*64:,} voxels')

model.train()
optimizer_lbfgs = torch.optim.LBFGS(
    model.parameters(), lr=1.0,
    max_iter=lbfgs_steps, line_search_fn='strong_wolfe', history_size=10,
)
lbfgs_hist = []

def lbfgs_closure():
    optimizer_lbfgs.zero_grad()
    loss = contraction_loss(model(T_lb, eps_lb), xi_lb)
    loss.backward()
    lbfgs_hist.append(float(loss))
    return loss

t_lb = time.time()
optimizer_lbfgs.step(lbfgs_closure)
t_lb = time.time() - t_lb

print(f'LBFGS done: {len(lbfgs_hist)} function evals in {t_lb:.1f}s')
print(f'  Loss: {lbfgs_hist[0]:.4e} → {lbfgs_hist[-1]:.4e}')

# Final val check after LBFGS
model.eval()
val_l = []
with torch.no_grad():
    for vb in loaders['val']:
        T_v, e_v, xi_v = prep_batch(vb)
        val_l.append(float(contraction_loss(model(T_v, e_v), xi_v)))
final_val = sum(val_l) / max(len(val_l), 1)
print(f'  Val loss after LBFGS: {final_val:.4e}')

# Save LBFGS checkpoint
lbfgs_ckpt = Path(RUN_DIR_FF) / 'freeform_kan_lbfgs.pt'
torch.save({'model_state_dict': model.state_dict(), 'config': cfg,
            'val_loss': final_val, 'lbfgs_evals': len(lbfgs_hist)}, lbfgs_ckpt)
print(f'  Checkpoint saved → {lbfgs_ckpt.name}')

In [ ]:
# §10 — Training curves
import matplotlib.pyplot as plt
import numpy as np

steps_all = [r['step']       for r in history]
train_all = [r['train_loss'] for r in history]
val_rows  = [(r['step'], r['val_loss']) for r in history if 'val_loss' in r]
val_s, val_l = zip(*val_rows) if val_rows else ([], [])

fig, ax = plt.subplots(figsize=(9, 4))
ax.semilogy(steps_all, train_all, 'b-', lw=0.8, alpha=0.6, label='train rel-L2')
if val_s:
    ax.semilogy(val_s, val_l, 'r--', lw=2, marker='o', ms=4, label='val rel-L2')
ax.axhline(1e-2, color='orange', ls=':', lw=1.5, label='1% threshold')
ax.axhline(1e-3, color='green',  ls=':', lw=1.5, label='0.1% threshold')
ax.set_xlabel('Adam step'); ax.set_ylabel('Relative L2 error'); ax.legend(fontsize=9)
ax.set_title('Free-form KAN — standalone regression loss')
ax.grid(True, alpha=0.3)
plt.tight_layout()
out = Path(RUN_DIR_FF) / 'freeform_kan_curves.png'
plt.savefig(out, dpi=150, bbox_inches='tight')
print(f'Saved to {out}')
plt.show()

## §11 — Bilinearity diagnostic: mixed Hessian ∂²ξ / ∂T ∂ε

For the exact bilinear target `ξ_i = Σ_j T_{ij} ε_j`, the mixed second-order
partial derivative is:

$$\frac{\partial^2 \xi_i}{\partial T_{jk} \, \partial \varepsilon_l} = \delta_{ij}\,\delta_{kl}$$

This is a **constant tensor** — independent of the input point `(T, ε)`.  
Two checks:  
1. **Constancy**: std across sample voxels / mean ≪ 1  
2. **Pattern**: cosine similarity to `δ_{ij}δ_{kl}` ≈ 1

> **Note on `eps_input_scale`:** the model scales ε by `eps_input_scale=100`
> before the KAN. This scales the mixed Hessian by 100 (see derivation in
> `FREEFORM_KAN_CONTRACTION_DISCOVERY.md §6.2`).  The pattern check (cosine)
> is unaffected; the constancy check is still valid.

In [ ]:
# §11 (code) — Compute mixed Hessian at several voxels

def mixed_hessian_at_voxel(model, T_vox, eps_vox):
    """
    T_vox:   (n, n) float64 — stiffness contrast at one voxel
    eps_vox: (n,)   float64 — strain at one voxel
    Returns: H (n, n, n, n) — ∂²ξ_i/∂T_{jk}∂ε_l
    """
    n = eps_vox.shape[0]
    T_in = T_vox.reshape(1, n, n, 1, 1).clone().requires_grad_(True)
    e_in = eps_vox.reshape(1, n, 1, 1).clone().requires_grad_(True)

    xi = model(T_in, e_in)   # (1, n, 1, 1)

    H = torch.zeros(n, n, n, n)
    for i in range(n):
        grad_T = torch.autograd.grad(
            xi[0, i, 0, 0], T_in, create_graph=True, retain_graph=True
        )[0]   # (1, n, n, 1, 1)
        for j in range(n):
            for kk in range(n):
                H_row = torch.autograd.grad(
                    grad_T[0, j, kk, 0, 0], e_in, retain_graph=True
                )[0]   # (1, n, 1, 1)
                H[i, j, kk, :] = H_row[0, :, 0, 0].detach()
    return H

# Expected Hessian for exact T:ε (after eps_input_scale factor):  eps_scale × δ_{ij}δ_{kl}
n_c = 3
eps_scale = float(model.eps_input_scale)
H_expected = torch.zeros(n_c, n_c, n_c, n_c)
for i in range(n_c):
    for kk in range(n_c):
        H_expected[i, i, kk, kk] = eps_scale

# Sample voxels from the val set
N_VOXELS = 6
torch.manual_seed(42)
val_batch = next(iter(loaders['val']))
T_val, eps_val, _ = prep_batch(val_batch)
B, n_c_, H_, W_ = eps_val.shape

model.eval()
H_samples = []
for _ in range(N_VOXELS):
    b  = torch.randint(0, B, (1,)).item()
    hh = torch.randint(0, H_, (1,)).item()
    ww = torch.randint(0, W_, (1,)).item()
    Tv  = T_val[b, :, :, hh, ww]
    ev  = eps_val[b, :, hh, ww]
    H_samples.append(mixed_hessian_at_voxel(model, Tv, ev))

H_stack = torch.stack(H_samples)         # (N, n, n, n, n)
H_mean  = H_stack.mean(0)
H_std   = H_stack.std(0)

constancy = H_std.norm() / H_mean.norm().clamp(min=1e-12)
cos_sim   = (H_mean.flatten() @ H_expected.flatten()) / (
                H_mean.norm() * H_expected.norm()).clamp(min=1e-12)

print('=' * 60)
print('Bilinearity Diagnostic')
print('=' * 60)
print(f'\nExpected pattern: H[i,j,k,l] = {eps_scale:.0f} × δ_ij × δ_kl')
print(f'\nMean Hessian (entries with |value| > {eps_scale*0.05:.1f}):')
threshold = eps_scale * 0.05
for i in range(n_c):
    for j in range(n_c):
        for kk in range(n_c):
            for ll in range(n_c):
                v  = H_mean[i, j, kk, ll].item()
                ex = H_expected[i, j, kk, ll].item()
                if abs(v) > threshold:
                    print(f'  H[{i},{j},{kk},{ll}] = {v:+7.2f}  expected {ex:+7.1f}')

print(f'\nConstancy (std/mean across {N_VOXELS} voxels): {constancy:.4f}')
print(f'  < 0.10  →  bilinear  ✓' if constancy < 0.10 else
      f'  > 0.10  →  higher-order terms present  ✗')
print(f'\nPattern cosine similarity to δ_ij δ_kl:  {cos_sim:.4f}')
print(f'  > 0.99  →  correct bilinear structure  ✓' if cos_sim > 0.99 else
      f'  < 0.99  →  structure differs from T:ε  ✗')

bilinear_ok = constancy < 0.10 and cos_sim > 0.99
print(f'\nOverall bilinearity verdict: {"PASS ✓" if bilinear_ok else "FAIL ✗"}')
print('=' * 60)

In [ ]:
# §12 — Embed in LSFNO and compare to FFT ground truth
#
# The FreeFormKANTauTheta has the same forward(T, eps) → xi signature as
# KANTauTheta, so it drops directly into LSFNO with no other changes.
# This tests whether the function learned from i.i.d. (T, ε) pairs
# generalises to the correlated (T, ε) pairs that the LS iteration visits.

from models.ls_fno import LSFNO
from utils.config_loader import load_config

model.eval()

# Build LSFNO with the free-form KAN as tau_theta
lsfno_cfg = load_config('configs/freeform_kan_colab.yaml')
lsfno_cfg['train_path'] = DATA_PATH
lsfno_ff = LSFNO.from_config(lsfno_cfg, tau_theta=model).to(device)
lsfno_ff.eval()

# Evaluate on test split
test_loader = torch.utils.data.DataLoader(
    DataLoaderFactory.from_config(cfg)['test'].dataset,
    batch_size=8, shuffle=False, num_workers=0
)

errors_ff = []
with torch.no_grad():
    for tb in test_loader:
        C  = tb['C_field'].to(device)
        eb = tb['eps_bar'].to(device)
        ep = tb['eps_star'].to(device)
        # use_checkpointing=False: free-form KAN forward is cheap, no need to checkpoint
        ep_pred = lsfno_ff(C, eb, use_checkpointing=False)
        diff  = (ep_pred - ep).reshape(ep.shape[0], -1)
        ref   = ep.reshape(ep.shape[0], -1)
        rel_l2 = (diff.norm(dim=1) / ref.norm(dim=1).clamp(min=1e-12)).cpu().numpy()
        errors_ff.extend(rel_l2.tolist())

import numpy as np
errors_ff = np.array(errors_ff)
print('=' * 55)
print('Embedded LSFNO (free-form KAN) vs FFT ground truth')
print('=' * 55)
print(f'Samples : {len(errors_ff)}')
print(f'Mean    : {errors_ff.mean():.4%}')
print(f'Median  : {np.median(errors_ff):.4%}')
print(f'p90     : {np.percentile(errors_ff, 90):.4%}')
print(f'Max     : {errors_ff.max():.4%}')
passed = errors_ff.mean() < 1e-3
print(f'Passed (mean < 0.1%): {"✓" if passed else "✗"}')
print('=' * 55)

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(errors_ff * 100, bins=30, color='steelblue', edgecolor='white', lw=0.4)
ax.axvline(errors_ff.mean() * 100, color='red',    ls='--', lw=1.5, label=f'Mean {errors_ff.mean():.3%}')
ax.axvline(0.1,                    color='orange', ls=':',  lw=1.5, label='0.1% threshold')
ax.set_xlabel('Relative L2 field error (%)'); ax.set_ylabel('Count')
ax.set_title('Embedded validation: free-form KAN vs FFT')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)
plt.tight_layout()
out2 = Path(RUN_DIR_FF) / 'freeform_kan_embedded_errors.png'
plt.savefig(out2, dpi=150, bbox_inches='tight')
plt.show()

---
## §13 — Results summary

Fill in after training completes:

| Criterion | Target | Measured |
|---|---|---|
| Final train rel-L2 | < 0.1% | |
| Final val rel-L2   | < 0.1% | |
| Embedded LSFNO mean error vs FFT | < 0.1% | |
| Bilinearity — constancy (std/mean) | < 0.10 | |
| Bilinearity — pattern cosine | > 0.99 | |
| Time for Adam (5,000 steps) | — | |
| Time for LBFGS | — | |

### Interpretation

| Result | Meaning |
|---|---|
| Low loss + bilinearity PASS | KAN discovered the bilinear structure from raw (T, ε) data ✓ |
| Low loss + bilinearity FAIL | KAN achieves accuracy but via a higher-order approximation — not a clean discovery |
| High loss | The architecture or training is insufficient; try depth-2 `width=[18,12]` |

**Comparison with Study 2 (scaffolded KAN):**  
Study 2 achieved sub-0.1% field error with 3 trainable parameters.  
This study uses ~2,430 parameters and trains a different objective (standalone ξ regression vs embedded ε regression).  
The gap in accuracy and interpretability directly quantifies the value of the polarization-identity architectural prior.

**Next:** If the bilinearity diagnostic passes, the follow-up is `shared=False`
(independent per-edge weights, 27 params in the scaffolded model) to check
whether the free-form KAN discovers the per-row connectivity structure.

In [ ]:
# §14 — Per-sample visualization: FFT ground truth vs Free-form KAN
#
# Can be run standalone after §1–§5 — does NOT require §6–§13 (no training).
# Loads the checkpoint directly from Drive, embeds the KAN in LSFNO, runs
# one full LS iteration on a single test sample, and plots a 4-row × 3-col
# colormap figure.
#
# Minimum cells to run first: §1 → §2 → §3 → §4 → §5
#
# Layout:
#   Row 0: Microstructure C₁₁₁₁(x)  |  Model / loading info  |  Rel-L2 summary
#   Row 1: FFT ground truth   ε₁₁ / ε₂₂ / 2ε₁₂
#   Row 2: Free-form KAN      ε₁₁ / ε₂₂ / 2ε₁₂  (same colour range as Row 1)
#   Row 3: Absolute error     ε₁₁ / ε₂₂ / 2ε₁₂  (sequential, own scale)
#
# Change SAMPLE_IDX to inspect different samples.

import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
import numpy as np
from datasets.micromechanics import MicromechanicsDataset
from models.ls_fno import LSFNO

# ── Choose checkpoint and sample ──────────────────────────────────────────────
# Both paths are derived from RUN_DIR_FF (defined in §3) — no dependency on §8.
_lbfgs_ckpt = Path(RUN_DIR_FF) / 'freeform_kan_lbfgs.pt'
_adam_ckpt  = Path(RUN_DIR_FF) / 'freeform_kan_checkpoint.pt'
VIS_CKPT    = _lbfgs_ckpt if _lbfgs_ckpt.exists() else _adam_ckpt

if not VIS_CKPT.exists():
    raise FileNotFoundError(
        f'No checkpoint found in {RUN_DIR_FF}.\n'
        f'Expected: freeform_kan_lbfgs.pt  or  freeform_kan_checkpoint.pt\n'
        f'Run §8 (and optionally §9) first to train and save the model.'
    )

SAMPLE_IDX = 0     # index within test split; change to inspect other samples
VIS_SPLIT  = 'test'
VIS_SAVE   = Path(RUN_DIR_FF) / f'freeform_vis_{VIS_SPLIT}_{SAMPLE_IDX}.png'

print(f'Checkpoint : {VIS_CKPT.name}')
print(f'Sample     : #{SAMPLE_IDX} from {VIS_SPLIT} split')

# ── Rebuild free-form model and load weights ──────────────────────────────────
_ckpt = torch.load(VIS_CKPT, map_location=device, weights_only=False)
vis_model = FreeFormKANTauTheta(
    n_comp          = 3,
    width           = cfg.get('width', [18]),
    grid_size       = int(cfg.get('grid_size', 5)),
    k               = int(cfg.get('k', 3)),
    x_range         = tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale = float(cfg.get('eps_input_scale', 100.0)),
).to(device)
vis_model.load_state_dict(_ckpt['model_state_dict'])
vis_model.eval()
print(f'Loaded     : {vis_model}')

# ── Embed in LSFNO ────────────────────────────────────────────────────────────
_lsfno_cfg = load_config('configs/freeform_kan_colab.yaml')
lsfno_vis  = LSFNO.from_config(_lsfno_cfg, tau_theta=vis_model).to(device)
lsfno_vis.eval()
print(f'LSFNO      : α₀={lsfno_vis.alpha_0:.4g}, K={lsfno_vis.K}')

# ── Load one test sample ──────────────────────────────────────────────────────
ds_test = MicromechanicsDataset(DATA_PATH, split=VIS_SPLIT)
if SAMPLE_IDX >= len(ds_test):
    raise IndexError(f'sample_idx={SAMPLE_IDX} >= split size {len(ds_test)}')

sample   = ds_test[SAMPLE_IDX]
sample_d = {k: (v.unsqueeze(0).to(device) if torch.is_tensor(v) else v)
            for k, v in sample.items()}

with torch.no_grad():
    eps_pred = lsfno_vis(
        sample_d['C_field'].float(),
        sample_d['eps_bar'].float(),
    )

# ── Numpy arrays ──────────────────────────────────────────────────────────────
eps_fft  = sample_d['eps_star'].squeeze(0).cpu().numpy()      # (3, N, N)  FFT truth
eps_kan  = eps_pred.squeeze(0).cpu().float().numpy()          # (3, N, N)  KAN prediction
C_field  = sample_d['C_field'].squeeze(0).cpu().numpy()       # (3, 3, N, N)
eps_bar  = sample_d['eps_bar'].squeeze(0).cpu().numpy()       # (3,)

error   = eps_kan - eps_fft
abs_err = np.abs(error)
ref_norm = np.linalg.norm(eps_fft.ravel())
rel_l2   = float(np.linalg.norm(error.ravel()) / max(ref_norm, 1e-12))

C11 = C_field[0, 0]   # C₁₁₁₁(x): high value = inclusion, low = matrix
COMP_LABELS = [r'$\varepsilon_{11}$', r'$\varepsilon_{22}$', r'$2\varepsilon_{12}$']

# ── Figure ────────────────────────────────────────────────────────────────────
fig = plt.figure(figsize=(14, 12))
gs  = gridspec.GridSpec(4, 3, figure=fig, hspace=0.45, wspace=0.35)

# Row 0, col 0: microstructure
ax_micro = fig.add_subplot(gs[0, 0])
im = ax_micro.imshow(C11, cmap='gray', origin='lower')
plt.colorbar(im, ax=ax_micro, fraction=0.046, pad=0.04)
ax_micro.set_title(r'Microstructure  $C_{1111}(x)$', fontsize=10)
ax_micro.axis('off')

# Row 0, col 1: loading / model info
ax_text = fig.add_subplot(gs[0, 1])
ax_text.axis('off')
loading_txt = (
    f'Applied macroscopic strain  ε̄\n'
    f'  ε₁₁  = {eps_bar[0]:+.5f}\n'
    f'  ε₂₂  = {eps_bar[1]:+.5f}\n'
    f'  2ε₁₂ = {eps_bar[2]:+.5f}\n'
    f'\nModel: FreeFormKANTauTheta\n'
    f'  dims   = {vis_model._dims}\n'
    f'  params = {vis_model.n_params():,}\n'
    f'  G={cfg.get("grid_size", 5)}, k={cfg.get("k", 3)}\n'
    f'  ε-scale = {vis_model.eps_input_scale}'
)
ax_text.text(0.05, 0.95, loading_txt, transform=ax_text.transAxes,
             fontsize=9, va='top', fontfamily='monospace',
             bbox=dict(boxstyle='round', facecolor='lightyellow', alpha=0.8))

# Row 0, col 2: summary scalar
ax_sum = fig.add_subplot(gs[0, 2])
ax_sum.axis('off')
verdict = 'PASS ✓' if rel_l2 < 1e-3 else f'FAIL — {rel_l2:.1%}'
summary_txt = (
    f'Relative L2 field error\n'
    f'‖ε_KAN − ε_FFT‖ / ‖ε_FFT‖\n\n'
    f'  {rel_l2:.4%}\n\n'
    f'  Threshold: 0.1%\n'
    f'  {verdict}'
)
color = 'lightgreen' if rel_l2 < 1e-3 else 'lightsalmon'
ax_sum.text(0.05, 0.95, summary_txt, transform=ax_sum.transAxes,
            fontsize=10, va='top', fontfamily='monospace',
            bbox=dict(boxstyle='round', facecolor=color, alpha=0.9))

# Rows 1–3: strain field colormaps
row_labels = ['FFT ground truth', 'Free-form KAN', 'Absolute error']
fields     = [eps_fft, eps_kan, abs_err]
cmaps      = ['RdBu_r', 'RdBu_r', 'Reds']

for row_i, (label, field, cmap) in enumerate(zip(row_labels, fields, cmaps)):
    for col_i in range(3):
        ax   = fig.add_subplot(gs[row_i + 1, col_i])
        comp = field[col_i]
        if row_i < 2:
            vmax = max(np.abs(eps_fft[col_i]).max(), 1e-10)
            vmin = -vmax
        else:
            vmin = 0.0
            vmax = max(abs_err[col_i].max(), 1e-10)
        im = ax.imshow(comp, cmap=cmap, origin='lower', vmin=vmin, vmax=vmax)
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        title = f'{label}\n{COMP_LABELS[col_i]}' if col_i == 0 else COMP_LABELS[col_i]
        ax.set_title(title, fontsize=9)
        ax.axis('off')

fig.suptitle(
    f'Free-form KAN vs FFT  (sample {SAMPLE_IDX} of {VIS_SPLIT} split)',
    fontsize=12, y=1.01,
)

plt.savefig(VIS_SAVE, dpi=150, bbox_inches='tight')
print(f'\nSaved → {VIS_SAVE}')
plt.show()
plt.close(fig)

print(f'Rel-L2 field error this sample: {rel_l2:.4%}')

In [ ]:
# §15 — Edge function visualization
#
# Can be run standalone after §1–§5 (no dependency on §8/§9).
# Plots every learned B-spline edge function φᵢⱼ(x) per layer.
# Robust to any depth (width=[18] depth-2 or width=[18,12] depth-3, etc.)
#
# Input layout (9 features — symmetric T reduction):
#   Features 0–5 : T upper-triangle row-major → T₀₀,T₀₁,T₀₂,T₁₁,T₁₂,T₂₂
#   Features 6–8 : ε₀×s, ε₁×s, ε₂×s   (s = eps_input_scale)
#
# What to look for (polarization identity signature):
#   • Linear edges (φ ≈ ax)   → model uses that input linearly (L1 sum/difference)
#   • Near-zero (‖φ‖ ≈ 0)    → connection is effectively pruned
#   • Quadratic (φ ≈ ax²)     → product representation via (a+b)²−(a−b)²=4ab
#   • S-shaped / asymmetric   → higher-order or boundary effects

import matplotlib.pyplot as plt
import numpy as np
import torch.nn.functional as F
from models.freeform_kan import _b_spline_basis

# ── Load checkpoint ───────────────────────────────────────────────────────────
_lbfgs_ckpt = Path(RUN_DIR_FF) / 'freeform_kan_lbfgs.pt'
_adam_ckpt  = Path(RUN_DIR_FF) / 'freeform_kan_checkpoint.pt'
_vis_ckpt   = _lbfgs_ckpt if _lbfgs_ckpt.exists() else _adam_ckpt
if not _vis_ckpt.exists():
    raise FileNotFoundError(f'No checkpoint in {RUN_DIR_FF}. Run §8 first.')

_c = torch.load(_vis_ckpt, map_location=device, weights_only=False)
vis_m = FreeFormKANTauTheta(
    n_comp=3, width=cfg.get('width', [18]),
    grid_size=int(cfg.get('grid_size', 5)), k=int(cfg.get('k', 3)),
    x_range=tuple(cfg.get('x_range', [-2.0, 2.0])),
    eps_input_scale=float(cfg.get('eps_input_scale', 100.0)),
).to(device)
vis_m.load_state_dict(_c['model_state_dict'])
vis_m.eval()

# ── Architecture summary ──────────────────────────────────────────────────────
n_G = int(cfg.get('grid_size', 5))
n_k = int(cfg.get('k', 3))
print(f'Model  : {vis_m}')
print(f'Dims   : {vis_m._dims}')
print()
print(f'{"Layer":<12} {"Edges":>8} {"Spline/edge":>12} {"SiLU/edge":>10} {"Params":>8}')
print('-' * 56)
total_edges = total_params = 0
for li, layer in enumerate(vis_m.layers):
    n_edges = layer.n_in * layer.n_out
    n_sp    = n_G + n_k
    n_par   = n_edges * (n_sp + 1)
    total_edges  += n_edges
    total_params += n_par
    tag = '← output' if li == len(vis_m.layers) - 1 else ''
    print(f'L{li+1}: {layer.n_in}→{layer.n_out:<5}  {n_edges:>8}  {n_sp:>12}  {"1":>10}  {n_par:>8,}  {tag}')
print('-' * 56)
print(f'{"Total":<12} {total_edges:>8}  {"—":>12}  {"—":>10}  {total_params:>8,}')

# ── Helper: evaluate one edge function φᵢⱼ(x) over x_vals ───────────────────
def eval_edge(layer, i, j, x_vals):
    x_t    = x_vals.unsqueeze(1).to(device).double()
    grid_i = layer.grid[i:i+1, :]
    basis  = _b_spline_basis(x_t, grid_i, layer.k)     # (M, 1, G+k)
    spline = (basis[:, 0, :] @ layer.spline_weight[j, i, :]).detach()
    base   = layer.base_weight[j, i].detach() * F.silu(x_vals.to(device).double())
    return (spline + base).cpu().float().numpy()

x_min  = float(vis_m.layers[0].grid[:, n_k].min())
x_max  = float(vis_m.layers[0].grid[:, n_G + n_k].max())
x_plot = torch.linspace(x_min, x_max, 200)

# ── Input feature labels for the first layer ──────────────────────────────────
_n   = 3   # n_comp
_s   = float(cfg.get('eps_input_scale', 100.0))
_T_labels   = [f'T{i}{j}' for i in range(_n) for j in range(i, _n)]  # 6 upper-tri
_eps_labels = [fr'ε{k}×{_s:.0f}' for k in range(_n)]                 # 3 ε
_input_labels = _T_labels + _eps_labels  # 9 total

# ── Inner (non-output) layers ─────────────────────────────────────────────────
inner_layers = vis_m.layers[:-1]
for li, layer in enumerate(inner_layers):
    n_in, n_out = layer.n_in, layer.n_out
    row_labels  = _input_labels if li == 0 else [f'h{i}' for i in range(n_in)]

    fig, axes = plt.subplots(
        n_in, n_out,
        figsize=(max(n_out * 0.85, 8), max(n_in * 0.85, 8)),
        sharex=True, sharey=False,
    )
    # Ensure axes is always 2-D
    if n_in == 1:
        axes = axes[np.newaxis, :]
    if n_out == 1:
        axes = axes[:, np.newaxis]

    fig.suptitle(
        f'Layer {li+1}: {n_in}→{n_out}  ({n_in*n_out} edge functions)\n'
        f'Row = input,  Col = hidden neuron j,  plot: φᵢⱼ(x)',
        fontsize=10,
    )
    for i in range(n_in):
        for j in range(n_out):
            ax = axes[i, j]
            y  = eval_edge(layer, i, j, x_plot)
            ax.plot(x_plot.numpy(), y, lw=0.7, color='steelblue')
            ax.axhline(0, color='k', lw=0.3, ls='--')
            ax.set_xticks([]); ax.set_yticks([])
            if j == 0:
                ax.set_ylabel(row_labels[i], fontsize=5, labelpad=1)
            if i == 0:
                ax.set_title(f'j={j}', fontsize=5, pad=1)
    plt.tight_layout()
    out_l = Path(RUN_DIR_FF) / f'freeform_kan_edges_L{li+1}.png'
    plt.savefig(out_l, dpi=150, bbox_inches='tight')
    print(f'Layer {li+1} edge grid saved → {out_l.name}')
    plt.show(); plt.close(fig)

# ── Output layer (always the last layer) ─────────────────────────────────────
out_layer      = vis_m.layers[-1]
out_li         = len(vis_m.layers)          # 1-indexed label
n_in_o, n_out_o = out_layer.n_in, out_layer.n_out

# Label inputs: feature names if depth-1 (first layer IS the only hidden→output),
# otherwise hidden neuron names
if len(vis_m.layers) == 1:
    in_labels_o = _input_labels[:n_in_o]
else:
    in_labels_o = [f'h{i}' for i in range(n_in_o)]
out_labels_o = [r'$\xi_0$', r'$\xi_1$', r'$\xi_2$']

fig_o, axes_o = plt.subplots(
    n_in_o, n_out_o,
    figsize=(n_out_o * 2.8, n_in_o * 1.6),
    sharex=True,
)
if n_in_o == 1:
    axes_o = axes_o[np.newaxis, :]
if n_out_o == 1:
    axes_o = axes_o[:, np.newaxis]

fig_o.suptitle(
    f'Layer {out_li} (output): {n_in_o}→{n_out_o}  ({n_in_o*n_out_o} edge functions)\n'
    r'Each plot: $\varphi_{ij}(x)$,  blue = mean≥0,  red = mean<0',
    fontsize=11,
)
for i in range(n_in_o):
    for j in range(n_out_o):
        ax  = axes_o[i, j]
        y   = eval_edge(out_layer, i, j, x_plot)
        col = 'steelblue' if y.mean() >= 0 else 'tomato'
        ax.plot(x_plot.numpy(), y, lw=1.2, color=col)
        ax.axhline(0, color='k', lw=0.5, ls='--')
        ax.set_yticks([])
        if i == n_in_o - 1:
            ax.set_xlabel(out_labels_o[j], fontsize=10)
            ax.tick_params(axis='x', labelsize=6)
        else:
            ax.set_xticks([])
        if j == 0:
            ax.set_ylabel(in_labels_o[i], fontsize=7, rotation=0, labelpad=22)
        l2 = float(np.sqrt(np.mean(y**2)))
        ax.text(0.97, 0.95, f'‖φ‖={l2:.2f}', transform=ax.transAxes,
                fontsize=5, ha='right', va='top', color='gray')

plt.tight_layout()
out_l_o = Path(RUN_DIR_FF) / f'freeform_kan_edges_L{out_li}_output.png'
plt.savefig(out_l_o, dpi=150, bbox_inches='tight')
print(f'Output layer edge grid saved → {out_l_o.name}')
plt.show(); plt.close(fig_o)

# ── Sparsity summary ──────────────────────────────────────────────────────────
print()
print('── Edge norm summary (‖φ‖ < 0.05 → near-zero / inactive) ──')
for li, layer in enumerate(vis_m.layers):
    norms = []
    for i in range(layer.n_in):
        for j in range(layer.n_out):
            y = eval_edge(layer, i, j, x_plot)
            norms.append(float(np.sqrt(np.mean(y**2))))
    norms    = np.array(norms)
    n_active = (norms >= 0.05).sum()
    print(f'  L{li+1}: {n_active}/{len(norms)} active edges  '
          f'({100*n_active/len(norms):.0f}%)  '
          f'max={norms.max():.3f}  median={np.median(norms):.3f}')
print()
print('Polarization identity would give (theory):')
print(f'  L1: 36/162 active (22%) with 2 edges per hidden neuron')
print(f'  L2 (output): 18/54 active (33%) with quadratic ±t²/4 shape')